### Notebook 08 — Final Response Agent

##### 1. Notebook Purpose

- The purpose of this notebook is to implement the Final Response Agent for the multi-agent customer support workflow.

- The Final Response Agent is responsible for generating the final customer-facing response by combining the validated outputs produced by the specialized agents.

- Unlike the SQL Agent, Prediction Agent, Vector Search Agent, and Retention Agent, this agent does not perform business analysis or decision-making. Instead, it synthesizes the available information into a single clear, concise, and grounded response.

- The generated response is validated, stored in the shared workflow state, and returned to the user.

##### 2. Technologies Used

- Python
- Databricks Notebooks
- Databricks Model Serving
- Large Language Models (LLMs)
- Prompt Engineering
- Pydantic
- TypedDict Shared State
- Multi-Agent Architecture

##### 3. Input

The Final Response Agent receives the current MultiAgentState.

The shared state may contain:

- Original user request
- Coordinator execution plan
- SQL Agent result
- Prediction Agent result
- Vector Search Agent result
- Retention Agent result
- Execution history
- Workflow errors

The exact information available depends on the execution plan created by the Coordinator Agent.

##### 4. Output

The Final Response Agent produces a validated FinalResponseAgentResult.

The result includes:

- Agent name
- Execution status
- Status message
- Task description
- Error details (when applicable)
- Final human-readable response

The agent updates the shared workflow state by storing:

- Final Response Agent result
- Final response
- Execution history
- Error information (if execution fails)

##### 5. Architecture

``` text

                      Customer Request
                             │
                             ▼
                    Coordinator Agent
                             │
                  Creates execution plan
                             │
      ┌──────────────┬───────────────┬──────────────┐
      ▼              ▼               ▼              ▼
 SQL Agent   Prediction Agent  Vector Search Agent  ...
      │              │               │
      └──────────────┴───────┬───────┘
                              ▼
                     Retention Agent
                              │
                              ▼
                  Final Response Agent
                              │
        Reads validated outputs from all agents
                              │
        Builds grounded prompt for the LLM
                              │
        Generates final customer response
                              │
                              ▼
                 Updates MultiAgentState
                 

```

##### 6. Load Shared Models and Helpers

In [0]:
%run ./01_shared_models

In [0]:
%run ./02_shared_state_and_helpers

##### 7. Imports

In [0]:
# Standard-library imports
import json
from typing import Any, Dict, Optional
from pydantic import ValidationError

# Databricks SDK
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ChatMessage,
    ChatMessageRole,
)

# initialize the workspace client
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# Foundation models
LLM_MODEL = "databricks-meta-llama-3-1-8b-instruct"


##### 8. Final Response System Prompt

In [0]:
FINAL_RESPONSE_SYSTEM_PROMPT = """
You are the Final Response Agent in a multi-agent customer support system.

Your responsibility is to convert validated agent results into one clear, concise, and professional response.

Follow these rules:

1. Use only the information supplied in the user request and validated agent results.

2. Do not invent facts, customer details, predictions, percentages, reasons, or recommendations.

3. Do not change a recommendation produced by the Retention Agent.

4. Do not recalculate SQL results or prediction confidence.

5. Clearly answer the original user request.

6. Use plain language that a customer support representative can understand.

7. Include important values such as counts, percentages, prediction labels, confidence values, and recommended actions when they are available.

8. Do not mention internal implementation details such as prompts, tools, schemas, shared state, routing, or agent execution.

9. If required information is unavailable, clearly explain that the available results are insufficient.

10. Do not include information from failed agent results as if it were valid.

11. Do not explain why an action might work unless that reasoning is explicitly included in the validated agent results.

Return only the final response. Do not return JSON, markdown code fences, or internal reasoning.
""".strip()

##### 9. Serialize One Agent Result

In [0]:
def serialize_agent_result(
    result: BaseAgentResult,
) -> Dict[str, Any]:
    """
    Convert one validated agent result into a
    JSON-compatible dictionary.
    """

    return result.model_dump(
        mode="json",
        exclude_none=True,
    )

##### 10. Collect Successful Agent Results

In [0]:
def collect_successful_agent_results(
    state: MultiAgentState,
) -> Dict[str, Dict[str, Any]]:
    """
    Collect successful specialized-agent results.

    The Final Response Agent's own previous result is excluded
    to prevent it from summarizing an older final response.
    """

    successful_results: Dict[
        str,
        Dict[str, Any],
    ] = {}

    agent_results = state.get(
        "agent_results",
        {},
    )

    for agent_name, result in agent_results.items():
        if agent_name == FINAL_RESPONSE_AGENT_NAME:
            continue

        if result.status != "success":
            continue

        successful_results[agent_name] = (
            serialize_agent_result(result)
        )

    return successful_results

##### 11. Build the Grounded Prompt

In [0]:
def build_final_response_prompt(
    state: MultiAgentState,
) -> str:
    """
    Build a grounded prompt using the original request and
    successful agent results.
    """

    user_request = state["user_request"]

    successful_results = (
        collect_successful_agent_results(state)
    )

    prompt_payload = {
        "user_request": user_request,
        "successful_agent_results": (
            successful_results
        ),
    }

    return (
        "Generate the final response using the following "
        "validated workflow information.\n\n"
        f"{json.dumps(prompt_payload, indent=2)}"
    )

##### 12. Extract Text from the Model Response

In [0]:
def extract_chat_response_text(
    response: Any,
) -> str:
    """
    Extract generated text from a Databricks chat endpoint
    response.

    Raises
    ------
    ValueError
        If the response does not contain generated text.
    """

    choices = getattr(
        response,
        "choices",
        None,
    )

    if not choices:
        raise ValueError(
            "The chat endpoint returned no choices."
        )

    first_choice = choices[0]

    message = getattr(
        first_choice,
        "message",
        None,
    )

    if message is None:
        raise ValueError(
            "The chat endpoint response did not contain "
            "a message."
        )

    content = getattr(
        message,
        "content",
        None,
    )

    if not isinstance(content, str):
        raise ValueError(
            "The chat endpoint response did not contain "
            "text content."
        )

    cleaned_content = content.strip()

    if not cleaned_content:
        raise ValueError(
            "The generated final response was empty."
        )

    return cleaned_content

##### 13. Databricks LLM endpoint

In [0]:
def invoke_final_response_llm(
    prompt: str,
) -> str:
    """
    Invoke the real Databricks LLM endpoint
    and return the generated text.
    """

    if not prompt or not prompt.strip():
        raise ValueError(
            "LLM prompt cannot be empty."
        )

    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt,
            )
        ],
        max_tokens=300,
        temperature=0.0,
    )

    if (
        not response.choices
        or response.choices[0].message is None
        or not response.choices[0].message.content
    ):
        raise ValueError(
            "The LLM returned no response."
        )

    return (
        response.choices[0]
        .message.content
        .strip()
    )

##### 14. Call the Language Model

In [0]:
def generate_final_response(
    state: MultiAgentState,
    llm_invoke: LLMInvokeFunction,
) -> str:
    """
    Generate a grounded final response using
    the injected LLM function.
    """

    user_prompt = build_final_response_prompt(
        state=state
    )

    full_prompt = f"""
{FINAL_RESPONSE_SYSTEM_PROMPT}

Grounded workflow information:

{user_prompt}
""".strip()

    final_response = llm_invoke(
        full_prompt
    )

    if not isinstance(final_response, str):
        raise TypeError(
            "The LLM must return a string."
        )

    final_response = final_response.strip()

    if not final_response:
        raise ValueError(
            "The generated final response is empty."
        )

    return final_response

##### 15. Execute final response agent

In [0]:
def execute_final_response_agent(
    state: MultiAgentState,
    task: AgentTask,
    llm_invoke: LLMInvokeFunction,
) -> FinalResponseAgentResult:
    """
    Execute the Final Response Agent's assigned task.
    """

    if task.agent_name != FINAL_RESPONSE_AGENT_NAME:
        raise ValueError(
            "The assigned task does not belong to "
            "final_response_agent."
        )

    successful_results = (
        collect_successful_agent_results(
            state=state
        )
    )

    if not successful_results:
        raise ValueError(
            "No successful agent results are available "
            "for final-response generation."
        )

    final_response = generate_final_response(
        state=state,
        llm_invoke=llm_invoke,
    )

    result_payload = {
        "agent_name": FINAL_RESPONSE_AGENT_NAME,
        "status": "success",
        "message": (
            "Final Response Agent completed successfully."
        ),
        "task_description": task.task_description,
        "error": None,
        "final_response": final_response,
    }

    return FinalResponseAgentResult.model_validate(
        result_payload
    )

##### 16. Main Final Response Agent

In [0]:
def run_final_response_agent(
    state: MultiAgentState,
    llm_invoke: LLMInvokeFunction,
) -> MultiAgentState:
    """
    Run the Final Response Agent inside the
    shared workflow.
    """

    try:
        coordinator_result = state.get(
            "coordinator_result"
        )

        if coordinator_result is None:
            raise ValueError(
                "Final Response Agent cannot run because "
                "coordinator_result is missing."
            )

        assigned_task = find_assigned_agent_task(
            coordinator_result=coordinator_result,
            agent_name=FINAL_RESPONSE_AGENT_NAME,
        )

        if assigned_task is None:
            record_agent_execution(
                state=state,
                agent_name=FINAL_RESPONSE_AGENT_NAME,
                status="skipped",
                message=(
                    "Final Response Agent was not required "
                    "by the execution plan."
                ),
            )

            return state

        validate_task_dependencies(
            state=state,
            task=assigned_task,
        )

        final_result = execute_final_response_agent(
            state=state,
            task=assigned_task,
            llm_invoke=llm_invoke,
        )

        store_agent_result(
            state=state,
            agent_result=final_result,
        )

        record_agent_execution(
            state=state,
            agent_name=FINAL_RESPONSE_AGENT_NAME,
            status="success",
            message=final_result.message,
        )

    except (
        ValueError,
        TypeError,
        KeyError,
        RuntimeError,
        ValidationError,
    ) as exc:

        error_message = str(exc)

        record_agent_execution(
            state=state,
            agent_name=FINAL_RESPONSE_AGENT_NAME,
            status="failed",
            message=(
                "Final Response Agent failed to "
                "generate the final response."
            ),
        )

        record_agent_error(
            state=state,
            agent_name=FINAL_RESPONSE_AGENT_NAME,
            error_code="FINAL_RESPONSE_AGENT_ERROR",
            error_message=error_message,
        )

    except Exception as exc:

        error_message = (
            "Unexpected Final Response Agent error: "
            f"{exc}"
        )

        record_agent_execution(
            state=state,
            agent_name=FINAL_RESPONSE_AGENT_NAME,
            status="failed",
            message=(
                "Final Response Agent encountered an "
                "unexpected error."
            ),
        )

        record_agent_error(
            state=state,
            agent_name=FINAL_RESPONSE_AGENT_NAME,
            error_code=(
                "FINAL_RESPONSE_AGENT_UNEXPECTED_ERROR"
            ),
            error_message=error_message,
        )

    return state

##### 17. Tests

In [0]:
def test_final_response_agent_unit_test() -> None:
    """
    Run deterministic unit tests for the
    Final Response Agent.

    Tests:
    1. Successful response from Retention result.
    2. Successful SQL-only response.
    3. Successful Prediction-only response.
    4. Final Response Agent skip behavior.
    5. Missing dependency.
    6. Empty LLM response.
    7. LLM execution failure.
    8. Failed agent results are excluded from grounding.
    """

    # =========================================================
    # Mock LLMs
    # =========================================================

    def mock_successful_llm(
        prompt: str,
    ) -> str:
        """
        Return a deterministic grounded response.
        """

        assert isinstance(prompt, str)
        assert prompt.strip()

        return (
            "The customer is predicted to be at risk "
            "of churn. A service-quality review is "
            "recommended based on the available "
            "customer information."
        )

    def mock_sql_llm(
        prompt: str,
    ) -> str:
        """
        Return a deterministic SQL-based response.
        """

        assert isinstance(prompt, str)
        assert prompt.strip()

        return (
            "There are 1,869 customers who churned."
        )

    def mock_prediction_llm(
        prompt: str,
    ) -> str:
        """
        Return a deterministic prediction response.
        """

        assert isinstance(prompt, str)
        assert prompt.strip()

        return (
            "Customer 7590-VHVEG is predicted "
            "to churn."
        )

    def mock_empty_llm(
        prompt: str,
    ) -> str:
        """
        Simulate an empty model response.
        """

        return "   "

    def mock_failed_llm(
        prompt: str,
    ) -> str:
        """
        Simulate an LLM execution failure.
        """

        raise RuntimeError(
            "Mock LLM endpoint unavailable."
        )

    # =========================================================
    # Coordinator helpers
    # =========================================================

    def create_final_response_coordinator_result(
        request_type: str,
        final_dependencies: List[AgentName],
    ) -> CoordinatorResult:
        """
        Create a Coordinator result containing a
        Final Response Agent task.
        """

        execution_plan = []

        for index, dependency in enumerate(
            final_dependencies,
            start=1,
        ):
            execution_plan.append(
                AgentTask(
                    task_id=f"task_{index}",
                    agent_name=dependency,
                    task_description=(
                        f"Execute {dependency} task."
                    ),
                    depends_on=[],
                )
            )

        execution_plan.append(
            AgentTask(
                task_id=(
                    f"task_{len(execution_plan) + 1}"
                ),
                agent_name=FINAL_RESPONSE_AGENT_NAME,
                task_description=(
                    "Generate the final grounded response."
                ),
                depends_on=final_dependencies,
            )
        )

        return CoordinatorResult(
            agent_name=COORDINATOR_AGENT_NAME,
            status="success",
            message=(
                "Execution plan created successfully."
            ),
            task_description=(
                "Plan the multi-agent workflow."
            ),
            error=None,
            request_type=request_type,
            reasoning=(
                "The request requires specialist "
                "processing followed by a grounded "
                "final response."
            ),
            execution_plan=execution_plan,
        )

    # =========================================================
    # Mock successful agent results
    # =========================================================

    def create_mock_prediction_result(
    ) -> PredictionAgentResult:
        """
        Create a successful Prediction Agent result.
        """

        return PredictionAgentResult(
            agent_name=PREDICTION_AGENT_NAME,
            status="success",
            message=(
                "Prediction task completed successfully."
            ),
            task_description=(
                "Predict churn for customer "
                "7590-VHVEG."
            ),
            error=None,
            customer_id="7590-VHVEG",
            predicted_category="Churn",
            confidence=None,
            model_name="mock_churn_model",
            raw_prediction=True,
        )

    def create_mock_retention_result(
    ) -> RetentionAgentResult:
        """
        Create a successful Retention Agent result.
        """

        return RetentionAgentResult(
            agent_name=RETENTION_AGENT_NAME,
            status="success",
            message=(
                "Retention task completed successfully."
            ),
            task_description=(
                "Recommend a retention action for "
                "customer 7590-VHVEG."
            ),
            error=None,
            task_id="task_3",
            customer_id="7590-VHVEG",
            recommended_action=(
                "service_quality_review"
            ),
            action_reason=(
                "The customer is predicted to churn "
                "and supporting information indicates "
                "service-quality concerns."
            ),
            prediction_label="Churn",
            prediction_confidence=None,
            supporting_notes=[
                (
                    "Customer reported repeated "
                    "connectivity problems."
                )
            ],
        )

    def create_mock_sql_result(
    ) -> SQLAgentResult:
        """
        Create a successful SQL Agent result.
        """

        return SQLAgentResult(
            agent_name=SQL_AGENT_NAME,
            status="success",
            message=(
                "SQL analytics task completed "
                "successfully."
            ),
            task_description=(
                "Count churned customers."
            ),
            error=None,
            sql_action="count_churned_customers",
            sql_result=[
                {
                    "churned_customers": 1869
                }
            ],
        )

    # =========================================================
    # TEST 1
    # Successful Retention final response
    # =========================================================

    print("=" * 80)
    print(
        "TEST 1: Successful retention final response"
    )
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        create_final_response_coordinator_result(
            request_type="retention",
            final_dependencies=[
                RETENTION_AGENT_NAME,
            ],
        )
    )

    state["agent_results"][
        RETENTION_AGENT_NAME
    ] = create_mock_retention_result()

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=mock_successful_llm,
    )

    print(updated_state)

    assert (
        FINAL_RESPONSE_AGENT_NAME
        in updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][FINAL_RESPONSE_AGENT_NAME]

    assert isinstance(
        result,
        FinalResponseAgentResult,
    )

    assert result.status == "success"

    assert result.final_response.strip()

    assert (
        "service-quality"
        in result.final_response.lower()
    )

    assert updated_state["errors"] == []

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "success"
    )

    print("PASS")
    print(result.model_dump())
    print()

    # =========================================================
    # TEST 2
    # SQL-only final response
    # =========================================================

    print("=" * 80)
    print("TEST 2: Successful SQL final response")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = (
        create_final_response_coordinator_result(
            request_type="sql_analytics",
            final_dependencies=[
                SQL_AGENT_NAME,
            ],
        )
    )

    state["agent_results"][
        SQL_AGENT_NAME
    ] = create_mock_sql_result()

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=mock_sql_llm,
    )

    result = updated_state[
        "agent_results"
    ][FINAL_RESPONSE_AGENT_NAME]

    assert result.status == "success"

    assert "1,869" in result.final_response

    assert updated_state["errors"] == []

    print("PASS")
    print(result.model_dump())
    print()

    # =========================================================
    # TEST 3
    # Prediction-only final response
    # =========================================================

    print("=" * 80)
    print(
        "TEST 3: Successful prediction final response"
    )
    print("=" * 80)

    state = create_initial_state(
        "Will customer 7590-VHVEG churn?"
    )

    state["coordinator_result"] = (
        create_final_response_coordinator_result(
            request_type="prediction",
            final_dependencies=[
                PREDICTION_AGENT_NAME,
            ],
        )
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=mock_prediction_llm,
    )

    result = updated_state[
        "agent_results"
    ][FINAL_RESPONSE_AGENT_NAME]

    assert result.status == "success"

    assert (
        "7590-vhveg"
        in result.final_response.lower()
    )

    assert "churn" in result.final_response.lower()

    assert updated_state["errors"] == []

    print("PASS")
    print(result.model_dump())
    print()

    # =========================================================
    # TEST 4
    # Final Response Agent skip
    # =========================================================

    print("=" * 80)
    print("TEST 4: Final Response Agent skips")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = (
        CoordinatorResult(
            agent_name=COORDINATOR_AGENT_NAME,
            status="success",
            message=(
                "Execution plan created successfully."
            ),
            task_description=(
                "Plan the workflow."
            ),
            error=None,
            request_type="sql_analytics",
            reasoning=(
                "Only SQL execution is included "
                "to verify skip behavior."
            ),
            execution_plan=[
                AgentTask(
                    task_id="task_1",
                    agent_name=SQL_AGENT_NAME,
                    task_description=(
                        "Count churned customers."
                    ),
                    depends_on=[],
                )
            ],
        )
    )

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=mock_successful_llm,
    )

    assert (
        FINAL_RESPONSE_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert updated_state["errors"] == []

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "skipped"
    )

    print("PASS")
    print(
        updated_state[
            "execution_history"
        ][-1]
    )
    print()

    # =========================================================
    # TEST 5
    # Missing required dependency
    # =========================================================

    print("=" * 80)
    print("TEST 5: Missing dependency")
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        create_final_response_coordinator_result(
            request_type="retention",
            final_dependencies=[
                RETENTION_AGENT_NAME,
            ],
        )
    )

    # Retention Agent result intentionally omitted.

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=mock_successful_llm,
    )

    assert (
        FINAL_RESPONSE_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "retention_agent"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()

    # =========================================================
    # TEST 6
    # Empty LLM response
    # =========================================================

    print("=" * 80)
    print("TEST 6: Empty LLM response")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = (
        create_final_response_coordinator_result(
            request_type="sql_analytics",
            final_dependencies=[
                SQL_AGENT_NAME,
            ],
        )
    )

    state["agent_results"][
        SQL_AGENT_NAME
    ] = create_mock_sql_result()

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=mock_empty_llm,
    )

    assert (
        FINAL_RESPONSE_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "empty"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()

    # =========================================================
    # TEST 7
    # LLM execution failure
    # =========================================================

    print("=" * 80)
    print("TEST 7: LLM execution failure")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = (
        create_final_response_coordinator_result(
            request_type="sql_analytics",
            final_dependencies=[
                SQL_AGENT_NAME,
            ],
        )
    )

    state["agent_results"][
        SQL_AGENT_NAME
    ] = create_mock_sql_result()

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=mock_failed_llm,
    )

    assert (
        FINAL_RESPONSE_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "unavailable"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()

    # =========================================================
    # TEST 8
    # Failed specialist result excluded
    # =========================================================

    print("=" * 80)
    print(
        "TEST 8: Failed agent result excluded "
        "from grounding"
    )
    print("=" * 80)

    state = create_initial_state(
        "Summarize the available customer information."
    )

    state["coordinator_result"] = (
        create_final_response_coordinator_result(
            request_type="combined",
            final_dependencies=[
                PREDICTION_AGENT_NAME,
            ],
        )
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    # Add a failed Vector Search result.
    # It is NOT a dependency of the final task and
    # should not be included in grounded results.

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = VectorSearchAgentResult(
        agent_name=VECTOR_SEARCH_AGENT_NAME,
        status="failed",
        message=(
            "Vector Search Agent failed."
        ),
        task_description=(
            "Search customer notes."
        ),
        error=(
            "Vector Search endpoint unavailable."
        ),
        task_id="task_2",
        query=None,
        results=[],
    )

    successful_results = (
        collect_successful_agent_results(
            state=state
        )
    )

    assert (
        PREDICTION_AGENT_NAME
        in successful_results
    )

    assert (
        VECTOR_SEARCH_AGENT_NAME
        not in successful_results
    )

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=mock_prediction_llm,
    )

    assert (
        FINAL_RESPONSE_AGENT_NAME
        in updated_state["agent_results"]
    )

    assert updated_state["errors"] == []

    print("PASS")
    print(
        updated_state[
            "agent_results"
        ][FINAL_RESPONSE_AGENT_NAME].model_dump()
    )
    print()

    print("=" * 80)
    print(
        "ALL FINAL RESPONSE AGENT UNIT TESTS PASSED"
    )
    print("=" * 80)

In [0]:
def test_final_response_agent_integration_test() -> None:
    """
    Test the Final Response Agent with the real
    Databricks language model.

    Specialist outputs are manually populated so
    this remains an integration test of the Final
    Response Agent rather than a full end-to-end test.
    """

    # =========================================================
    # Final Response task
    # =========================================================

    final_task = AgentTask(
        task_id="task_4",
        agent_name=FINAL_RESPONSE_AGENT_NAME,
        task_description=(
            "Generate the final grounded response."
        ),
        depends_on=[
            RETENTION_AGENT_NAME,
        ],
    )

    coordinator_result = CoordinatorResult(
        agent_name=COORDINATOR_AGENT_NAME,
        status="success",
        message=(
            "Execution plan created successfully."
        ),
        task_description=(
            "Plan the multi-agent workflow."
        ),
        error=None,
        request_type="retention",
        reasoning=(
            "The request requires a retention "
            "recommendation followed by a grounded "
            "final response."
        ),
        execution_plan=[
            final_task,
        ],
    )

    # =========================================================
    # Initial shared state
    # =========================================================

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        coordinator_result
    )

    # =========================================================
    # Validated Retention Agent result
    # =========================================================

    retention_result = RetentionAgentResult(
        agent_name=RETENTION_AGENT_NAME,
        status="success",
        message=(
            "Retention task completed successfully."
        ),
        task_description=(
            "Recommend a retention action for "
            "customer 7590-VHVEG."
        ),
        error=None,
        task_id="task_3",
        customer_id="7590-VHVEG",
        recommended_action=(
            "service_quality_review"
        ),
        action_reason=(
            "The customer is predicted to churn "
            "and the supporting notes indicate "
            "service quality concerns."
        ),
        prediction_label="Churn",
        prediction_confidence=None,
        supporting_notes=[
            (
                "Customer reported repeated "
                "connectivity problems."
            )
        ],
    )

    state["agent_results"][
        RETENTION_AGENT_NAME
    ] = retention_result

    # =========================================================
    # Run Final Response Agent with real LLM
    # =========================================================

    updated_state = run_final_response_agent(
        state=state,
        llm_invoke=invoke_final_response_llm,
    )

    # =========================================================
    # Display results
    # =========================================================

    print("Final Response Agent Result:")

    print(
        updated_state[
            "agent_results"
        ].get(FINAL_RESPONSE_AGENT_NAME)
    )

    print("\nExecution History:")

    print(
        updated_state[
            "execution_history"
        ]
    )

    print("\nErrors:")

    print(
        updated_state[
            "errors"
        ]
    )

    # =========================================================
    # Assertions
    # =========================================================

    assert (
        FINAL_RESPONSE_AGENT_NAME
        in updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][FINAL_RESPONSE_AGENT_NAME]

    assert isinstance(
        result,
        FinalResponseAgentResult,
    )

    assert result.status == "success"

    assert isinstance(
        result.final_response,
        str,
    )

    assert result.final_response.strip()

    assert updated_state["errors"] == []

    assert (
        updated_state[
            "execution_history"
        ][-1].agent_name
        == FINAL_RESPONSE_AGENT_NAME
    )

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "success"
    )

    print(
        "\nFinal Response:"
    )

    print(
        result.final_response
    )

    print(
        "\nFinal Response Agent "
        "integration test passed."
    )

##### 18. Expected Output

The exact wording may vary slightly depending on the language model, but the response should remain grounded in the supplied agent results.

Example:

- Customer 1001 has a high likelihood of churning, with aprediction confidence of 91%.

- The available customer notes indicate repeated internet outages, slow speeds, and frustration with technical support delays.

- The recommended retention action is to offer a support package to address the customer's ongoing technical concerns.

##### 19. Key Learnings

- The Final Response Agent does not perform new business analysis.

- It synthesizes validated results produced by specialized agents.

- Dependency validation prevents the agent from using missing  or failed upstream results.

- Only successful agent results are included in the grounded  prompt.

- Prompt instructions help prevent unsupported claims and  accidental changes to business recommendations.

- `FinalResponseAgentResult` provides a validated and consistent output structure.

- The final response is stored separately in  `state["final_response"]` so applications can access it  easily.

- Execution history and structured error records improve  observability and debugging.

##### 20. Notebook Conclusion

- In this section, we implemented the Final Response Agent for the multi-agent customer support workflow.

- The agent reads validated outputs from the specialized agents, builds a grounded prompt, invokes a Databricks-hosted language model, validates the generated response, and stores the final
  answer in the shared workflow state.

- The agent preserves separation of responsibilities by avoiding SQL analysis, churn prediction, semantic retrieval, and retention decision-making.

- With the Final Response Agent complete, all major agents needed for the multi-agent system are now available.

##### 21. Next Notebook

Part 9 — End-to-End Multi-Agent Orchestration

The next section will connect all agents into one complete workflow.

The orchestrator will:

- Receive the user request
- Create the initial shared state
- Execute the Coordinator Agent
- Read the execution plan
- Run specialized agents in dependency order
- Update shared state after every agent
- Handle failures and skipped tasks
- Execute the Final Response Agent
- Return the final grounded response